# 08 — GPT v2: the improved model

A tour of the `llm/` package. See `docs/09_v2_improvements.md` for the reasoning
behind each change.

1. a byte-level BPE tokenizer trained from scratch,
2. the modern architecture: RoPE, RMSNorm, SwiGLU, grouped-query attention, fused
   attention and weight tying,
3. the shipped checkpoints (pretrained and chat-finetuned) and the KV cache speed-up,
4. the ablation results.

In [1]:
import os, sys, time, json
sys.path.insert(0, os.path.abspath('..'))
import torch
from llm.tokenizer import BPETokenizer, CharTokenizer
from llm.model import GPT, GPTConfig, rope_cache, apply_rope
from llm.checkpoint import load_checkpoint
torch.manual_seed(0)

## 1. Byte-level BPE, trained from scratch

In [2]:
with open('../data/wizard_of_oz.txt', encoding='utf-8') as f:
    text = f.read()
train_text = text[: int(0.8 * len(text))]

t = time.time()
bpe = BPETokenizer.train(train_text, vocab_size=512)
print(f'trained {len(bpe.merges)} merges in {time.time() - t:.2f}s; vocab = {bpe.vocab_size}')
print('first merges:', [bpe.decode([256 + i]) for i in range(12)])
print('late merges: ', [bpe.decode([256 + i]) for i in range(len(bpe.merges) - 12, len(bpe.merges))])

sentence = 'Dorothy and the Wizard escaped from the Gargoyles!'
ids = bpe.encode(sentence)
print(len(sentence), 'chars ->', len(ids), 'tokens:', [bpe.decode([i]) for i in ids])
assert bpe.decode(bpe.encode('any text, even 世界 🙂')) == 'any text, even 世界 🙂'   # bytes: never <unk>

trained 253 merges in 0.19s; vocab = 512
first merges: [' t', 'he', ' a', ' the', 're', 'in', ' s', ' w', 'nd', 'ou', 'ed', ' b']
late merges:  [' sh', 'ers', ' ro', ' E', ' M', ' tim', '                ', ' pe', 'uggy', 'ther', 'us', 'ings']
50 chars -> 18 tokens: ['Dorothy', ' and', ' the', ' Wizard', ' ', 'es', 'ca', 'p', 'ed', ' from', ' the', ' ', 'G', 'ar', 'g', 'oy', 'les', '!']


In [3]:
for v in [512, 1024, 2048]:
    tok = bpe if v == 512 else BPETokenizer.train(train_text, v)
    val = text[int(0.8 * len(text)):]
    print(f'vocab {v:5d}: {len(val) / len(tok.encode(val)):.2f} characters per token on held-out text')

vocab   512: 2.04 characters per token on held-out text
vocab  1024: 2.59 characters per token on held-out text


vocab  2048: 3.04 characters per token on held-out text


## 2. The architecture

**RoPE** rotates queries and keys by a position-dependent angle. After rotation, the
dot product depends only on the *distance* between two tokens:

In [4]:
cos, sin = rope_cache(64, 16, 10000.0)
q, k = torch.randn(1, 1, 1, 16), torch.randn(1, 1, 1, 16)
rot = lambda v, p: apply_rope(v, cos[p:p + 1], sin[p:p + 1])
for i, j in [(3, 1), (13, 11), (50, 48), (10, 1)]:
    print(f'q at {i:2d}, k at {j:2d} (distance {i - j}): q·k = {(rot(q, i) * rot(k, j)).sum():+.4f}')

q at  3, k at  1 (distance 2): q·k = -1.2793
q at 13, k at 11 (distance 2): q·k = -1.2793
q at 50, k at 48 (distance 2): q·k = -1.2793
q at 10, k at  1 (distance 9): q·k = -6.1902


In [5]:
cfg = GPTConfig(vocab_size=512, block_size=128, n_layer=4, n_head=4, n_embd=128)
model = GPT(cfg)
print(model.blocks[0])
print(f'{sum(p.numel() for p in model.parameters()) / 1e6:.2f}M parameters '
      f'({model.num_params() / 1e6:.2f}M excluding the tied embedding)')
gqa = GPT(GPTConfig(vocab_size=512, block_size=128, n_layer=4, n_head=4, n_kv_head=1, n_embd=128))
print(f'with grouped-query attention (1 KV head): {sum(p.numel() for p in gqa.parameters()) / 1e6:.2f}M, KV cache 4x smaller')

Block(
  (norm1): RMSNorm()
  (attn): CausalSelfAttention(
    (qkv): Linear(in_features=128, out_features=384, bias=False)
    (proj): Linear(in_features=128, out_features=128, bias=False)
    (resid_dropout): Dropout(p=0.1, inplace=False)
  )
  (norm2): RMSNorm()
  (mlp): MLP(
    (gate): Linear(in_features=128, out_features=352, bias=False)
    (up): Linear(in_features=128, out_features=352, bias=False)
    (down): Linear(in_features=352, out_features=128, bias=False)
    (dropout): Dropout(p=0.1, inplace=False)
  )
)
0.87M parameters (0.80M excluding the tied embedding)
with grouped-query attention (1 KV head): 0.77M, KV cache 4x smaller


## 3. Shipped checkpoints

`models/oz-base.pt` is pretrained on the book (BPE-512, 4 layers, 128 dims, dropout 0.3,
weight decay 0.5). `models/oz-chat.pt` is the same model after instruction finetuning on
`data/oz_sft.jsonl`. Checkpoints hold only tensors and plain data, so they load with
`weights_only=True`.

In [6]:
base, tok, ckpt = load_checkpoint('../models/oz-base.pt')
print(base.config)
torch.manual_seed(42)
prompt = 'Dorothy looked at the Wizard and said'
x = torch.tensor([tok.encode(prompt)])
out = base.generate_all(x, 150, temperature=0.8, top_k=50, top_p=0.95, repetition_penalty=1.1)
print(tok.decode(out[0].tolist()))

GPTConfig(vocab_size=512, block_size=128, n_layer=4, n_head=4, n_kv_head=4, n_embd=128, dropout=0.3, bias=False, pos_emb='rope', norm='rms', mlp='swiglu', tie_weights=True, rope_theta=10000.0)


Dorothy looked at the Wizard and said:

"What's about it is so very folks in this low in her great single entranced
that such day to be came to the mountain. But they got to the passage had
fastened, for a little little time, or the carries of their head and began to walked
his around the earth, until the other cab-horse struck his face here with
the chance of this girl.

As the


### KV cache speed-up
Without a cache every new token re-runs the whole context. With a cache only the new
token passes through the network. The gap grows with context length and model size.

In [7]:
x = torch.tensor([tok.encode('The Wizard')])
for use_cache in [False, True]:
    torch.manual_seed(0)
    t = time.time()
    base.generate_all(x, 400, temperature=0, use_cache=use_cache)
    print(f'use_cache={use_cache}: {400 / (time.time() - t):.0f} tokens/s')

use_cache=False: 224 tokens/s


use_cache=True: 545 tokens/s


### Chat model
The prompt goes through the chat template `<|user|>…<|assistant|>` and generation stops
at `<|endoftext|>`. The model has about 1M parameters and was trained on a single book,
so it recalls answers it saw in finetuning (including paraphrases) but cannot reason
about new questions. That needs far more data and parameters (see
`docs/07_pretraining_vs_finetuning.md`).

In [8]:
from llm.generate import chat_prompt, USER, EOT
chat, tok, _ = load_checkpoint('../models/oz-chat.pt')
stop = {tok.special_tokens[EOT], tok.special_tokens[USER]}
for question in ['Who is Jim?', 'What do the Gargoyles fear most?', 'Can you tell me about the Braided Man?',
                 'How did Ozma rescue Dorothy?']:
    torch.manual_seed(0)
    x = torch.tensor([tok.encode(chat_prompt([], question))])
    out = chat.generate_all(x, 80, temperature=0.3, stop_ids=stop)
    answer = tok.decode([i for i in out[0, x.shape[1]:].tolist() if i not in stop])
    print(f'Q: {question}\nA: {answer}\n')

Q: Who is Jim?
A: Jim is the cab-horse. He is old and very thin, but he can talk once they reach the fairy countries.

Q: What do the Gargoyles fear most?
A: What the Gargoyles dread most is a noise.

Q: Can you tell me about the Braided Man?
A: The Braided Man lives in Pyramid Mountain. His hair and his beard are braided, and he makes Assorted Flutters and Rustles.



Q: How did Ozma rescue Dorothy?
A: Ozma used the Magic Belt to wish Dorothy and her friends out of the earth and into the Land of Oz.



## 4. Ablation: what each change buys
All runs: 4 layers, 4 heads, 128 dims, 2000 steps, CPU. Numbers are bits per character
on the full validation text (lower is better), from `experiments/ablation.sh`.

In [9]:
print(open('../experiments/results_ablation.md').read())

| run | data | params | steps | val loss/token | val bits/char | time |
|---|---|---|---|---|---|---|
| 3_v2_arch_char | oz_char | 0.81M | 2000 | 1.435 | **2.070** | 14.1 min |
| 2_v1_arch_v2_recipe | oz_char | 0.83M | 2000 | 1.465 | **2.113** | 12.7 min |
| 4_v2_bpe512 | oz_bpe512 | 0.87M | 2000 | 3.004 | **2.124** | 14.5 min |
| 5_v2_bpe1024 | oz_bpe1024 | 0.94M | 2000 | 3.823 | **2.130** | 15.5 min |
| 6_v2_bpe2048 | oz_bpe2048 | 1.07M | 2000 | 4.489 | **2.133** | 17.6 min |
| 1_v1_baseline | oz_char | 0.82M | 2000 | 1.685 | **2.431** | 6.3 min |

